# SAVERouter: a CPU-friendly tutorial

This notebook builds a small structured routing problem, acquires exactly $K$ model outcomes per training query, fits SAVERouter, and measures when the supervision investment pays back. It uses synthetic data and runs without benchmark downloads or external model APIs.

For the paper results, use the [full reproduction workflow](https://github.com/LAMDA-Model-Reuse/SaveRouter#reproducing-the-experiments).

In [ ]:
%pip -q install "saverouter==0.1.0" matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from saverouter import (
    FixedKSparseRouter,
    simulate_fixed_k_supervision,
    supervision_amortized_metrics,
)

rng = np.random.default_rng(42)

## 1. Create a structured routing problem

The synthetic pool has 12 candidate models. The first three are economical specialists for three query groups; later models are increasingly capable generalists but cost more.

In [ ]:
n_train, n_test, n_models, n_features = 300, 900, 12, 12
model_costs = np.linspace(0.12, 1.00, n_models)
model_costs[:3] = [0.28, 0.31, 0.34]
context_weights = rng.normal(0, 0.012, size=(n_features, n_models))

def make_split(n_queries):
    groups = rng.integers(0, 3, size=n_queries)
    features = rng.normal(0, 0.35, size=(n_queries, n_features))
    features[np.arange(n_queries), groups] += 2.0
    quality = 0.50 + 0.015 * np.arange(n_models)[None, :]
    quality = np.broadcast_to(quality, (n_queries, n_models)).copy()
    for specialist in range(3):
        quality[:, specialist] += np.where(groups == specialist, 0.35, -0.08)
    quality += features @ context_weights
    quality += rng.normal(0, 0.012, size=quality.shape)
    rewards = np.clip(quality, 0.0, 1.0)
    costs = np.broadcast_to(model_costs, rewards.shape).copy()
    return features, groups, rewards, costs

X_train, groups_train, rewards_train, costs_train = make_split(n_train)
X_test, groups_test, rewards_test, costs_test = make_split(n_test)
print(f"Train matrix: {rewards_train.shape}; candidate feedback pairs: {rewards_train.size:,}")

## 2. Acquire sparse feedback

SAVERouter selects exactly four candidate models for each training query. The resulting supervision artifact contains only the acquired query-model outcomes.

In [ ]:
feedback = simulate_fixed_k_supervision(
    rewards_train,
    groups_train,
    costs=costs_train,
    k=4,
    seed=42,
)

print(f"Acquired outcomes: {feedback.n_observations:,}")
print(f"Feedback density: {feedback.density:.1%}")
print(f"Exactly K observations per query: {np.unique(feedback.observation_counts).tolist()}")

## 3. Fit and evaluate the router

We fit from the sparse artifact and route test queries under a per-query cost cap. The best single model supplies the target-quality reference used by the amortized metrics.

In [ ]:
router = FixedKSparseRouter().fit(X_train, feedback)
choices = router.route(X_test, max_cost=0.40, group_ids=groups_test)

best_single = int(np.argmax(rewards_test.mean(axis=0)))
baseline_quality = float(rewards_test[:, best_single].mean())
baseline_cost = float(model_costs[best_single])
routed_quality = float(rewards_test[np.arange(n_test), choices].mean())
routed_cost = float(model_costs[choices].mean())
upfront_cost = float(feedback.costs.sum())

print(f"Best single model: {best_single} | quality={baseline_quality:.3f} | cost={baseline_cost:.3f}")
print(f"SAVERouter:        quality={routed_quality:.3f} | cost={routed_cost:.3f}")
print(f"Upfront supervision expenditure: {upfront_cost:.2f}")

## 4. Measure payback

SA-BEP is the deployment volume needed to recover the upfront supervision expenditure. SA-CR@H is the cost ratio after amortizing that expenditure over a horizon of $H$ requests.

In [ ]:
horizon = 10_000
metrics = supervision_amortized_metrics(
    target_achieved=routed_quality >= baseline_quality,
    baseline_cost_per_request=baseline_cost,
    routed_cost_per_request=routed_cost,
    supervision_cost=upfront_cost,
    horizon=horizon,
)
metrics

In [ ]:
traffic = np.linspace(0, horizon, 300)
baseline_total = traffic * baseline_cost
router_total = upfront_cost + traffic * routed_cost

plt.figure(figsize=(8.5, 4.6))
plt.plot(traffic, baseline_total, label="Best single model", linewidth=2.5)
plt.plot(traffic, router_total, label="SAVERouter + supervision", linewidth=2.5)
if np.isfinite(metrics["SA-BEP"]):
    plt.axvline(metrics["SA-BEP"], color="0.35", linestyle="--", label=f"Break-even: {metrics['SA-BEP']:,}")
plt.xlabel("Deployment queries")
plt.ylabel("Cumulative cost")
plt.title("When does sparse supervision pay back?")
plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()
plt.show()

## Next steps

- Run the repository's download-free test with `saverouter smoke-test`.
- Adapt [`examples/online_feedback.py`](https://github.com/LAMDA-Model-Reuse/SaveRouter/blob/main/examples/online_feedback.py) to acquire feedback from real model calls.
- Reproduce the paper's four benchmarks with `bash scripts/reproduce.sh all`.

The numerical values in this tutorial are illustrative and are not paper benchmark results.